# Screenshot Categorization

Two datasets with distinct roles:

1. **Labeled slice** - 218 images manually tagged via the `www/` labeling tool. Used for
   supervised evaluation of OCR and CLIP classifiers.
2. **Full corpus** - all screenshots resolved from `config.yaml`. Used for OCR caching, CLIP
   clustering, and co-occurrence analysis against manual labels.

Assumes `config.yaml` is configured, `samples.py` has been run, and the labeling tool has been
used to produce `data/notebook/labels.jsonl`.

### Vocabulary

- **Label** - manually assigned category from the `www/` labeling tool.
- **Feature** - numerical signal used by a model: TF-IDF weights, CLIP embedding dimensions.
- **Embedding** - dense vector representation of an image. CLIP base32 uses 512 dims.
- **Prediction** - model output for one or more labels.

### Objectives

- Test whether OCR text features can carry the screenshot label space.
- Compare OCR and CLIP on the same labeled slice.
- Explore CLIP clustering on the full corpus.
- Surface cluster slices to accelerate manual labeling in `www/`.

In [ ]:
import json
import sqlite3
import time
from collections import Counter
from pathlib import Path

import hdbscan
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import umap
from PIL import Image
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.svm import LinearSVC

from bruki.config import load_config, resolve_paths


def stopwatch(flag):
    if flag:
        stopwatch.s = time.perf_counter()
    else:
        print(time.perf_counter() - stopwatch.s)


stopwatch(1)

Labeled rows are loaded from `labels.jsonl`. `EXCLUDE_LABELS` filters categories too noisy or
ambiguous to evaluate. Full corpus paths are resolved from config for OCR caching and
full-corpus CLIP inference.


In [ ]:
ROOT           = Path.cwd()
LABELS_PATH    = ROOT / "data/notebook/labels.jsonl"
EXCLUDE_PATH   = ROOT / "data/notebook/exclude.txt"
EXCLUDE_LABELS = set(EXCLUDE_PATH.read_text(encoding="utf-8").splitlines())

rows = [json.loads(line) for line in LABELS_PATH.read_text(
    encoding="utf-8").splitlines() if line.strip()]
paths = [ROOT / row["input_path"] for row in rows]

config = load_config(str(ROOT / "config.yaml"))
path_groups = resolve_paths(
    config,
    series=config.plots["screenshots"].series,
    prefix="screenshot",
)
all_image_paths = [path for _, _, source_paths in path_groups for path in source_paths]
all_image_paths = list(dict.fromkeys(all_image_paths))
print(f"resolved {len(all_image_paths)} images")

`make_df` wraps classifier output into a DataFrame indexed by filename. `top_k` returns the
k highest-scoring images for a given label. `top_labels` picks the n highest-mass columns by
summing probability across all images. `split_indices` produces a reproducible train/test split.
`ensure_label_coverage` moves examples from test to train when a label has no training
representation. `best_global_threshold` sweeps a threshold grid to maximize micro-F1.

In [ ]:
SEED      = 42
TEST_SIZE = 0.2
THRESHOLD = 0.2


def make_df(probs, binarizer):
    return pd.DataFrame(probs, columns=binarizer.classes_, index=[p.name for p in paths])


def top_k(df, label, k=12):
    scores = df[label].to_numpy()
    order = np.argsort(scores)[::-1][:k]
    return [{"path": paths[i], "score": scores[i]} for i in order]


def top_labels(df, n=5):
    return df.sum().nlargest(n).index.tolist()


def print_hits(hits):
    for h in hits:
        print(f"{h['score']:.4f}  {h['path'].parent.name}/{h['path'].name}")


def plot_hits(hits, columns=4, label=""):
    rows_ = (len(hits) + columns - 1) // columns
    fig, axes = plt.subplots(rows_, columns, figsize=(3 * columns, 3 * rows_))
    axes = np.array(axes).reshape(-1)
    for ax, hit in zip(axes, hits):
        ax.imshow(Image.open(hit["path"]))
        ax.set_title(f"{hit['score']:.3f}\n{hit['path'].name}")
        ax.axis("off")
    for ax in axes[len(hits):]:
        ax.axis("off")
    if label:
        fig.suptitle(label)
    plt.tight_layout()
    plt.show()


def split_indices(n):
    idx = np.random.RandomState(SEED).permutation(n)
    cut = int(n * (1 - TEST_SIZE))
    return idx[:cut], idx[cut:]


def ensure_label_coverage(targets, train_idx, test_idx):
    train_set, test_set = set(train_idx.tolist()), set(test_idx.tolist())
    for col in range(targets.shape[1]):
        if targets[list(train_set), col].sum() > 0:
            continue
        candidates = [i for i in test_set if targets[i, col] == 1]
        if candidates:
            test_set.remove(candidates[0])
            train_set.add(candidates[0])
    return np.array(sorted(train_set)), np.array(sorted(test_set))


def numeric_density(text):
    return sum(c.isdigit() for c in text) / max(len(text), 1)

In [ ]:
def split_train_val(train_idx, val_size=0.2):
    train_idx = np.array(train_idx, dtype=int)
    rng       = np.random.RandomState(SEED)
    perm      = rng.permutation(train_idx)
    cut       = max(1, int(len(perm) * val_size))
    val       = np.array(sorted(perm[:cut]), dtype=int)
    train     = np.array(sorted(perm[cut:]), dtype=int)
    return train, val


def best_global_threshold(targets, probs, average="micro"):
    grid       = np.linspace(0.01, 0.30, 60)
    best_t     = THRESHOLD
    best_score = -1.0
    for t in grid:
        preds = (probs >= t).astype(int)
        score = f1_score(targets, preds, average=average, zero_division=0)
        if score > best_score:
            best_score = score
            best_t = float(t)
    return best_t, best_score

### Label Distribution

Labels with fewer than 5 examples are excluded from classifier evaluation--too few examples
for a reliable train/test split. `viable` is the working label set for all downstream models.


In [ ]:
label_counts = Counter(label for r in rows for label in r["categories"])
s = pd.Series(label_counts).sort_values()
print(f"total labels: {len(s)}")
print(f"labels with <5  examples:  {(s < 5).sum()}")
print(f"labels with <10 examples:  {(s < 10).sum()}")
print(f"labels with >=5 examples:  {(s >= 5).sum()}")
print(f"labels with >=10 examples: {(s >= 10).sum()}")

viable = s[s >= 5].index.tolist()
print(f"\nviable labels: {viable}")

## Text Model

### OCR Baseline

OCR is evaluated as a baseline because screenshots often contain text. The question is whether
lexical content alone carries enough signal to classify screenshot intent.

Expected strengths: lexical cues (`receipt`, URLs, shell commands), numeric patterns (prices,
scores, timestamps), repeated UI text fragments. Expected weaknesses: labels that are
visual/structural rather than lexical, images with no readable text, and Tesseract noise on
graphical elements.

This section is a viability test. If OCR macro-F1 stays low on the viable label set, OCR is
treated as auxiliary metadata rather than a primary representation.

OCR extraction is performed once in `ocr.ipynb` and read here from `ocr.sqlite3`. All OCR text
is lowercased at write time.


In [ ]:
DB_PATH = ROOT / "data/notebook/ocr.sqlite3"

conn = sqlite3.connect(DB_PATH)
ocr_by_path = dict(conn.execute("SELECT input_path, text FROM ocr_doc"))
conn.close()

print(f"loaded OCR cache rows: {len(ocr_by_path)}")

OCR text for the labeled slice is aligned to `paths` by exact string key match against the
cache.


In [ ]:
text_ocr = [ocr_by_path[str(p)] for p in paths]

### Corpus: per-label OCR character profile

Per-label word count, numeric density, and empty-document rate. Empty is defined as fewer than
20 characters after stripping. This surfaces which labels are OCR-hostile before any modeling.

In [ ]:
for label in viable:
    indices   = [i for i, r in enumerate(rows) if label in r["categories"]]
    subset    = [text_ocr[i] for i in indices]
    lengths   = [len(t.split()) for t in subset]
    densities = [numeric_density(t) for t in subset]
    empty = sum(1 for t in subset if len(t.strip()) < 20)
    print(
        f"{label:20} n={len(indices):3}  mean_words={np.mean(lengths):6.0f}  "
        f"mean_density={np.mean(densities):.3f}  empty={empty}"
    )

### Top word tokens per viable label

Raw word frequency per label using a count vectorizer over the labeled slice. Diagnostic only —
shows which lexical cues are available per category before TF-IDF weighting.

In [ ]:
cv    = CountVectorizer(analyzer="word", min_df=1)
X_cv  = cv.fit_transform(text_ocr)
vocab = np.array(cv.get_feature_names_out())

for label in viable:
    indices = [i for i, r in enumerate(rows) if label in r["categories"]]
    sums = np.asarray(X_cv[indices].sum(axis=0)).squeeze()
    top = np.argsort(sums)[::-1][:15]
    print(f"\n--- {label} ---")
    print(", ".join(f"{vocab[i]}({sums[i]:.0f})" for i in top))

### Label separability

Cosine similarity between `statistics` and `monitorat` under char and word TF-IDF. These two
labels share a visual context (system monitoring) so high similarity here would indicate the
text feature space cannot separate them. Computed before the classifier sweep to set
expectations.

In [ ]:
char_vec  = TfidfVectorizer(analyzer="char", ngram_range=(3, 5), max_df=0.95)
word_vec  = TfidfVectorizer(analyzer="word", min_df=1, max_df=0.95)
char_feat = char_vec.fit_transform(text_ocr)
word_feat = word_vec.fit_transform(text_ocr)

stats_idx   = [i for i, r in enumerate(rows) if "statistics" in r["categories"]]
monitor_idx = [
    i
    for i, r in enumerate(rows)
    if "monitorat" in r["categories"] and "statistics" not in r["categories"]
]

for name, feat in [("char", char_feat), ("word", word_feat)]:
    if not stats_idx or not monitor_idx:
        print(
            f"{name}  statistics vs monitorat: skipped "
            f"(statistics={len(stats_idx)}, monitorat={len(monitor_idx)})"
        )
        continue

    sim = cosine_similarity(feat[stats_idx], feat[monitor_idx])
    print(f"{name}  statistics vs monitorat: mean={sim.mean():.3f} max={sim.max():.3f}")

### Vectorizer + classifier sweep

Grid search over four TF-IDF configurations and two classifiers on the viable label subset.
Numeric density is tested as an additive feature. Restricted to images with at least one viable
label. `ensure_label_coverage` prevents any label from being absent from the training split.
Micro and macro F1 are both reported--macro penalizes low-support labels that micro averaging
would wash out.

In [ ]:
viable_set    = set(viable)
viable_mask   = np.array([any(label in viable_set for label in r["categories"]) for r in rows])
viable_idx    = np.where(viable_mask)[0]

sub_rows      = [rows[i] for i in viable_idx]
sub_ocr       = [text_ocr[i] for i in viable_idx]
sub_binarizer = MultiLabelBinarizer(classes=viable)
sub_targets   = sub_binarizer.fit_transform(
    [[label for label in r["categories"] if label in viable_set] for r in sub_rows])
sub_density   = np.array([numeric_density(t) for t in sub_ocr]).reshape(-1, 1)

tr, te        = ensure_label_coverage(sub_targets, *split_indices(len(sub_rows)))

vectorizers = {
    "char(3,5)": TfidfVectorizer(analyzer="char", ngram_range=(3, 5), max_df=0.95),
    "char(2,4)": TfidfVectorizer(analyzer="char", ngram_range=(2, 4), max_df=0.95),
    "word": TfidfVectorizer(analyzer="word", min_df=1, max_df=0.95),
    "word+bi": TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=1, max_df=0.95),
}

classifiers = {
    "logreg": OneVsRestClassifier(LogisticRegression(max_iter=1000, solver="liblinear")),
    "svc": OneVsRestClassifier(LinearSVC(max_iter=2000)),
}

for vec_name, vec in vectorizers.items():
    feat = vec.fit_transform(sub_ocr)
    feat_den = np.hstack([feat.toarray(), sub_density])
    for clf_name, clf in classifiers.items():
        for feat_label, f in [(vec_name, feat), (f"{vec_name}+density", feat_den)]:
            clf.fit(f[tr], sub_targets[tr])
            raw = (
                clf.predict_proba(f[te])
                if hasattr(clf, "predict_proba")
                else (clf.decision_function(f[te]) >= 0).astype(int)
            )
            preds = (raw >= THRESHOLD).astype(int) if hasattr(clf, "predict_proba") else raw
            micro = f1_score(sub_targets[te], preds, average="micro", zero_division=0)
            macro = f1_score(sub_targets[te], preds, average="macro", zero_division=0)
            print(f"{clf_name:8} {feat_label:30} micro={micro:.3f} macro={macro:.3f}")

### Winning configuration

Trains and saves the sweep winner. Char n-gram (2,4) TF-IDF with logistic regression. Model is
persisted to `data/notebook/models/text.joblib` for use in downstream comparison against CLIP.

In [ ]:
text_binarizer = MultiLabelBinarizer()
text_targets   = text_binarizer.fit_transform([r["categories"] for r in rows])

text_vec       = TfidfVectorizer(analyzer="char", ngram_range=(2, 4), max_df=0.95)
text_features  = text_vec.fit_transform(text_ocr)

tr, te         = ensure_label_coverage(text_targets, *split_indices(len(rows)))
text_clf       = OneVsRestClassifier(LogisticRegression(max_iter=1000, solver="liblinear"))

text_clf.fit(text_features[tr], text_targets[tr])

text_probs     = text_clf.predict_proba(text_features)
text_df        = make_df(text_probs, text_binarizer)

joblib.dump(
    {"vectorizer": text_vec, "classifier": text_clf, "binarizer": text_binarizer},
    ROOT / "data/notebook/models/text.joblib",
)

### Survey

Top-5 labels by aggregate model confidence, with image galleries for spot-checking. Commented
out by default--run selectively.

In [ ]:
for label in text_df[top_labels(text_df)]:
    plot_hits(top_k(text_df, label, k=8), label=label)

## Image Model

ResNet18 (ImageNet pretrained) was evaluated and rejected. Cosine similarity between visually
distinct label pairs scored 0.56-0.70, indicating the embedding space does not separate
screenshot categories. ResNet's pretraining domain is natural photographs; all screenshots
collapse into a single undifferentiated region.

CLIP (`openai/clip-vit-base-patch32`) was pretrained contrastively on web-scraped image-text
pairs including UI, screenshots, and documents. The same label pairs score 0.24-0.45 under
CLIP--a meaningful improvement in separability. `pooler_output` from the vision encoder is
projected via `visual_projection` to a 512-dim embedding. Fixed resolution, no variable
patching, CPU-viable at this corpus size.

In [ ]:
CLIP_MODEL = "openai/clip-vit-base-patch32"
CLIP_CACHE = ROOT / f"data/notebook/models/{CLIP_MODEL.replace('/', '__')}.pkl"

clip_cached           = joblib.load(CLIP_CACHE)
clip_cache_paths      = clip_cached["paths"]
clip_cache_index      = {path: i for i, path in enumerate(clip_cache_paths)}
clip_cache_embeddings = np.asarray(clip_cached["embeddings"], dtype=np.float32)
clip_cache_valid_mask = np.asarray(clip_cached["valid_mask"], dtype=bool)

row_positions         = np.array([clip_cache_index[str(path)] for path in paths], dtype=int)
clip_embeddings       = clip_cache_embeddings[row_positions]
clip_valid_mask       = clip_cache_valid_mask[row_positions]
clip_embeddings_valid = clip_embeddings[clip_valid_mask]

CLIP embeddings for the labeled slice are loaded directly from the shared cache written by
`clusters.ipynb`. Positions are resolved by path string lookup into the cache index. Invalid
rows (failed image load) are masked out before any classifier work.

### Features

CLIP embeddings aligned to the labeled slice. Dropped count reflects images that failed
`prepare_image` at embed time.

In [ ]:
image_rows      = [r for i, r in enumerate(rows) if clip_valid_mask[i]]
image_binarizer = MultiLabelBinarizer()
image_targets   = image_binarizer.fit_transform([r["categories"] for r in image_rows])

print(
    f"samples: {len(image_rows)} (dropped={int((~clip_valid_mask).sum())})  "
    f"labels:  {len(image_binarizer.classes_)}  "
    f"dims:    {clip_embeddings.shape[1]}"
)

### Training

A validation split is carved from the training pool to calibrate the decision threshold via
`best_global_threshold`. The model is then refit on the full training pool before test
evaluation. Both micro and macro F1 are reported with a per-label classification report.

In [ ]:
tr_full, te = ensure_label_coverage(image_targets, *split_indices(len(image_rows)))
tr, va      = split_train_val(tr_full, val_size=0.2)
tr, va      = ensure_label_coverage(image_targets, tr, va)

image_clf = OneVsRestClassifier(LogisticRegression(max_iter=1000, solver="liblinear"))
image_clf.fit(clip_embeddings_valid[tr], image_targets[tr])

image_probs_val = image_clf.predict_proba(clip_embeddings_valid[va])
IMAGE_THRESHOLD, val_micro = best_global_threshold(
    image_targets[va], image_probs_val, average="micro")
print("image threshold:", f"{IMAGE_THRESHOLD:.3f}")
print("val micro:", f"{val_micro:.3f}")

image_clf.fit(clip_embeddings_valid[tr_full], image_targets[tr_full])

image_probs_test = image_clf.predict_proba(clip_embeddings_valid[te])
image_preds_test = (image_probs_test >= IMAGE_THRESHOLD).astype(int)

print("micro:", f1_score(image_targets[te], image_preds_test, average="micro", zero_division=0))
print("macro:", f1_score(image_targets[te], image_preds_test, average="macro", zero_division=0))
print(
    classification_report(
        image_targets[te], image_preds_test, target_names=image_binarizer.classes_, zero_division=0
    )
)

### Score and save

Full-slice probabilities and the calibrated threshold are persisted to
`data/notebook/models/image.joblib` alongside the binarizer and the row index mapping back to
the original labeled set.


In [ ]:
image_probs = image_clf.predict_proba(clip_embeddings_valid)
image_df    = make_df(image_probs, image_binarizer)

joblib.dump(
    {
        "classifier": image_clf,
        "binarizer": image_binarizer,
        "threshold": IMAGE_THRESHOLD,
        "row_indices": np.where(clip_valid_mask)[0],
    },
    ROOT / "data/notebook/models/image.joblib",
)

In [ ]:
image_df[top_labels(image_df)].head(5)

In [ ]:
# for label in ["hockey", "browser", "food"]:
for label in top_labels(image_df):
    plot_hits(top_k(image_df, label, k=8), label=label)

## CLIP vs OCR: Controlled Evaluation

Both pipelines are evaluated on the same viable-label subset under identical train/test splits
so the comparison is controlled. Thresholds are calibrated independently on the same validation
split.

- **Text**:  char n-gram (2,4) TF-IDF + logistic regression on OCR transcripts
- **Image**: logistic regression on CLIP embeddings

The per-label `delta` table (image_f1 - text_f1) is the primary decision surface. If CLIP materially outperforms OCR across the viable label set, CLIP becomes the default representation for clustering and labeling.

### Side-by-Side Per-Label F1 on Viable Set

Both models are evaluated on the same viable subset and aligned split strategy so the deltas are
directly comparable label-by-label.

In [ ]:
def build_synthesis_dataset():
    eval_idx  = np.where(viable_mask & clip_valid_mask)[0]
    eval_rows = [rows[i] for i in eval_idx]
    eval_ocr  = [text_ocr[i] for i in eval_idx]
    eval_clip = clip_embeddings[eval_idx]

    binarizer    = MultiLabelBinarizer(classes=viable)
    eval_targets = binarizer.fit_transform(
        [[label for label in row["categories"] if label in viable_set] for row in eval_rows]
    )

    train_full, test = ensure_label_coverage(eval_targets, *split_indices(len(eval_rows)))
    train, val       = split_train_val(train_full, val_size=0.2)
    train, val       = ensure_label_coverage(eval_targets, train, val)

    return {
        "rows": eval_rows,
        "ocr": eval_ocr,
        "clip": eval_clip,
        "targets": eval_targets,
        "train": train,
        "train_full": train_full,
        "val": val,
        "test": test,
    }


syn = build_synthesis_dataset()

#### Text Model on Shared Split

OCR text is modeled with the winning text configuration from the earlier sweep; thresholding is calibrated on validation and then refit for test-time comparison.

In [ ]:
def fit_text_synthesis_model(data):
    text_vec  = TfidfVectorizer(analyzer="char", ngram_range=(2, 4), max_df=0.95)
    text_feat = text_vec.fit_transform(data["ocr"])
    text_clf  = OneVsRestClassifier(LogisticRegression(max_iter=1000, solver="liblinear"))
    text_clf.fit(text_feat[data["train"]], data["targets"][data["train"]])

    val_probs = text_clf.predict_proba(text_feat[data["val"]])
    threshold, val_micro = best_global_threshold(
        data["targets"][data["val"]], val_probs, average="micro"
    )
    print("text threshold:", f"{threshold:.3f}")
    print("text val micro:", f"{val_micro:.3f}")

    text_clf.fit(text_feat[data["train_full"]], data["targets"][data["train_full"]])
    return (text_clf.predict_proba(text_feat[data["test"]]) >= threshold).astype(int)


text_test_preds = fit_text_synthesis_model(syn)

#### Image Model on Shared Split
CLIP embeddings use the same train/validation/test structure as text so gains/losses are attributable to representation, not evaluation setup.


In [ ]:
def fit_image_synthesis_model(data):
    image_clf = OneVsRestClassifier(LogisticRegression(max_iter=1000, solver="liblinear"))
    image_clf.fit(data["clip"][data["train"]], data["targets"][data["train"]])

    val_probs = image_clf.predict_proba(data["clip"][data["val"]])
    threshold, val_micro = best_global_threshold(
        data["targets"][data["val"]], val_probs, average="micro"
    )
    print("image threshold (viable):", f"{threshold:.3f}")
    print("image val micro (viable):", f"{val_micro:.3f}")

    image_clf.fit(data["clip"][data["train_full"]], data["targets"][data["train_full"]])
    return (image_clf.predict_proba(data["clip"][data["test"]]) >= threshold).astype(int)


image_test_preds = fit_image_synthesis_model(syn)

### Per-Label F1 Comparison

Use the `delta = image_f1 - text_f1` table as the decision surface:
- positive delta: CLIP advantage
- negative delta: OCR advantage

The aggregate micro/macro lines summarize overall direction; the per-label table identifies where the exceptions live.

In [ ]:
text_f1s  = f1_score(syn["targets"][syn["test"]], text_test_preds, average=None, zero_division=0)
image_f1s = f1_score(syn["targets"][syn["test"]], image_test_preds, average=None, zero_division=0)
support   = syn["targets"][syn["test"]].sum(axis=0)

cmp = pd.DataFrame(
    {
        "text_f1": text_f1s,
        "image_f1": image_f1s,
        "delta": image_f1s - text_f1s,
        "support": support.astype(int),
    },
    index=viable,
).sort_values("delta", ascending=False)

text_micro  = f1_score(syn["targets"][syn["test"]], text_test_preds,
                       average="micro", zero_division=0)
text_macro  = f1_score(syn["targets"][syn["test"]], text_test_preds,
                       average="macro", zero_division=0)
image_micro = f1_score(syn["targets"][syn["test"]], image_test_preds,
                       average="micro", zero_division=0)
image_macro = f1_score(syn["targets"][syn["test"]], image_test_preds,
                       average="macro", zero_division=0)

print(cmp.to_string(float_format="{:.3f}".format))
print()
print(f"text micro/macro:  {text_micro:.3f} / {text_macro:.3f}")
print(f"image micro/macro: {image_micro:.3f} / {image_macro:.3f}")
print(f"delta micro:       {image_micro - text_micro:+.3f}")
print(f"delta macro:       {image_macro - text_macro:+.3f}")

### Performance Comparison

Throughput is loaded from precomputed CSVs written by `ocr.ipynb` and `clusters.ipynb`. CLIP
is filtered to `clip-vit-base-patch32` for a like-for-like comparison against Tesseract. Both
sec/image and est. full-run time are included. In these CPU benchmarks, **CLIP-base32 is materially
faster than Tesseract**, reinforcing it as the primary pipeline on throughput grounds.

In [ ]:
clip = pd.read_csv(ROOT / "data/notebook/perf/embed.csv")
ocr = pd.read_csv(ROOT / "data/notebook/perf/ocr.csv")

perf = pd.concat(
    [clip.query("model == 'openai/clip-vit-base-patch32'"), ocr],
    ignore_index=True,
)
perf = perf[["model", "sec_per_image", "est_full_min"]].sort_values("sec_per_image")
perf["speed_vs_fastest_x"] = (perf["sec_per_image"] / perf["sec_per_image"].min()).round(2)
perf[["model", "speed_vs_fastest_x", "sec_per_image", "est_full_min"]]

### Conclusions: OCR vs CLIP

Two conclusions hold from the outputs in this section:

1. **Representation quality:** CLIP embeddings are the stronger primary feature space on the supervised slice (higher overall micro/macro behavior than OCR text features).
2. **Operational cost:** CLIP is also materially faster than OCR in the benchmark table, so the better representation is also the cheaper one at corpus scale.

OCR is still useful, but as a secondary signal. It recovers label signal in text-dominant cases that
CLIP can miss, while CLIP carries the broader default load.

## CLIP Clusters for Labeling Workflow

Given the synthesis and throughput comparisons above, full-corpus CLIP embeddings are clustered
to surface structure across the unlabeled majority. The
cluster view is intended to accelerate labeling in `www/`:

1. Open a cluster slice in gallery mode.
2. Multi-select obvious in-cluster matches.
3. Excise outliers with quick deselect edits.

This keeps clustering on the representation with the best combined accuracy/cost profile, while leaving OCR available as an auxiliary layer for text-heavy edge cases and possibly fuzzy tag sugestions.

## Jaccard Similarity of Cluster OCR Vocabulary vs Manual Labels

Interpretability aid for unlabeled clusters. OCR token sets are built per cluster (union of all
member image OCR) and per manual label (union across labeled images). Jaccard overlap between
the two gives a candidate label hint for each cluster without requiring visual inspection of
every image.

High overlap is a signal worth investigating, not a classification decision. Final assignment
still requires visual confirmation in the labeler gallery. Labels in `EXCLUDE_LABELS` are
filtered from the heatmap. Clusters where no label exceeds `THRESHOLD` (0.10) are dropped from
the column axis.

The clustermap rows are sorted by cluster size descending. Each row label includes cluster id,
member count, and dominant label by max Jaccard. `vmax` is capped at the 99th percentile to
prevent outlier bleaching.


### 1) Map Each Image Path to Its Cluster ID

In [ ]:
all_positions = np.array([clip_cache_index[str(path)] for path in all_image_paths], dtype=int)
all_clip_embeddings = clip_cache_embeddings[all_positions]
all_clip_valid_mask = clip_cache_valid_mask[all_positions]

cluster_paths = [path for path, ok in zip(all_image_paths, all_clip_valid_mask) if ok]
cluster_embeddings = all_clip_embeddings[all_clip_valid_mask]

reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    random_state=SEED,
)
embedding2d = reducer.fit_transform(cluster_embeddings)

clusterer = hdbscan.HDBSCAN(min_cluster_size=10, min_samples=5, metric="euclidean")
cluster_labels = clusterer.fit_predict(embedding2d)

assert len(cluster_paths) == len(cluster_labels)
path_to_cluster = {cluster_paths[i]: int(cluster_labels[i]) for i in range(len(cluster_paths))}

### 2) Token Sets per Cluster: Union of All OCR Words Across Member Images

In [ ]:
def tokenize(text):
    return set(text.lower().split())


cluster_vocab = {}
for i, path in enumerate(cluster_paths):
    cid = cluster_labels[i]
    text = ocr_by_path.get(str(path))
    if text is None:
        continue
    tokens = tokenize(text)
    cluster_vocab.setdefault(cid, set()).update(tokens)

### 3) Token Sets per Label: Union of OCR Words Across Labeled Images

In [ ]:
label_vocab = {}
for r, ocr in zip(rows, text_ocr):
    tokens = tokenize(ocr)
    for label in r["categories"]:
        label_vocab.setdefault(label, set()).update(tokens)

### 4) Compare Jaccard Similarity: Cluster vs Label

In [ ]:
def jaccard(a, b):
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


unique_clusters = sorted(c for c in set(cluster_labels) if c != -1)
labels_sorted   = sorted(label_vocab)

jac = pd.DataFrame(
    [
        [
            jaccard(cluster_vocab.get(cid, set()), label_vocab.get(label, set()))
            for label in labels_sorted
        ]
        for cid in unique_clusters
    ],
    index=[f"c{cid}" for cid in unique_clusters],
    columns=labels_sorted,
)

# jac

### 5) Show Top 5 Label Matches per Cluster

In [ ]:
def cluster_match_table(jac, cluster_labels, unique_clusters, top_n=5):
    rows = {}
    for cid in unique_clusters:
        n = (cluster_labels == cid).sum()
        top = jac.loc[f"c{cid}"].nlargest(top_n)
        row = {"n": n}
        for rank, (label, score) in enumerate(top.items(), 1):
            row[f"top{rank}"] = f"{label} ({score:.3f})"
        rows[f"c{cid}"] = row

    return pd.DataFrame(rows).T


cluster_match_table(jac, cluster_labels, unique_clusters)

### 6) Heatmap: Column Filter and Drop Labels When No Cluster Scores Highly

This makes it clear, numerically, how diffuse the labels are for each cluster. It is too close to call if top2 is statistically better than he top3 label for c0 (the c0 cluster is pictures of people's faces with little to no OCR data)

It's better to get a visual on the diffussion, which is where a heatmap is the right show.

In [ ]:
THRESHOLD   = 0.10
VMAX_PCTILE = 99  # prevents outlier bleaching

active_cols   = [
    col for col in jac.columns
    if jac[col].max() >= THRESHOLD and col not in EXCLUDE_LABELS
]
excluded_cols = [col for col in jac.columns if col in EXCLUDE_LABELS]
cluster_sizes = {cid: int((cluster_labels == cid).sum()) for cid in unique_clusters}
ROW_ORDER     = sorted(unique_clusters, key=lambda cid: cluster_sizes[cid], reverse=True)
if excluded_cols:
    ROW_ORDER = [
        cid for cid in ROW_ORDER
        if jac.loc[f"c{cid}", excluded_cols].max() < THRESHOLD
    ]
COL_ORDER = jac[active_cols].max(axis=0).sort_values(ascending=False).index.tolist()

J        = jac.reindex(index=[f"c{cid}" for cid in ROW_ORDER],
                       columns=COL_ORDER, fill_value=0).copy()
dominant = J.idxmax(axis=1).where(J.max(axis=1) > 0,
                                  "none") if COL_ORDER else pd.Series("none", index=J.index)
J.index  = [
    f"c{i.lstrip('c'):>2} n={cluster_sizes[int(i.lstrip('c'))]:>3} [{dominant[i]}]" for i in J.index
]

# J.shape, len(active_cols)
# J.iloc[:5, :5]

### 7) Plot Filtered OCR-Jaccard Clustermap

In [ ]:
vmax = np.percentile(J.values, VMAX_PCTILE)

cg = sns.clustermap(
    J,
    method="ward",
    metric="euclidean",
    cmap="YlOrRd",
    vmin=0,
    vmax=vmax,
    figsize=(max(14, len(COL_ORDER) * 0.45), max(12, len(ROW_ORDER) * 0.22)),
    linewidths=0.2,
    linecolor="#e0e0e0",
    xticklabels=True,
    yticklabels=True,
    row_cluster=False,
    col_cluster=False,
    dendrogram_ratio=(0.10, 0.06),
    cbar_pos=(0.01, 0.84, 0.02, 0.12),
    cbar_kws={"label": f"Jaccard (capped p{VMAX_PCTILE})"},
)
plt.setp(cg.ax_heatmap.get_xticklabels(), rotation=45, ha="right", fontsize=8)
plt.setp(cg.ax_heatmap.get_yticklabels(), fontsize=7.5)
cg.ax_heatmap.set_xlabel("Label", fontsize=10)
cg.ax_heatmap.set_ylabel("")
cg.ax_col_dendrogram.set_title(
    f"Cluster × Label - Ward/Euclidean clustermap  (threshold={THRESHOLD})", fontsize=11
)
plt.show()

# J.shape, vmax

## CLIP Cluster Vote Transfer

For each labeled image, the nearest non-self match in the full-corpus embedding space is found
by cosine similarity above `SIM_MIN`. The cluster assignment of that match is treated as a
vote. Aggregating votes per labeled row and cluster produces `clip_vote`: a labeled-row ×
cluster matrix of vote share. This is a soft label propagation mechanism--labeled images
recruit cluster support from their nearest embedding neighbors without requiring an explicit
classifier.

`MIN_CLUSTER_ROWS` filters clusters with too few labeled-row matches to be meaningful.
`FOCUS_LABELS` and `FOCUS_CLUSTERS` cap the heatmap dimensions for readability.

### 1) Set Parameters and Aligned Slices

In [ ]:
SIM_MIN          = 0.98
MIN_CLUSTER_ROWS = 2
FOCUS_LABELS     = 40
FOCUS_CLUSTERS   = 48

labeled_valid_idx = np.where(clip_valid_mask)[0]
labeled_rows      = [rows[i] for i in labeled_valid_idx]
labeled_text_ocr  = [text_ocr[i] for i in labeled_valid_idx]
labeled           = clip_embeddings[labeled_valid_idx].astype(float)
full              = cluster_embeddings.astype(float)
full_clusters     = np.asarray(cluster_labels, dtype=int)

# (len(labeled_rows), labeled.shape, full.shape, full_clusters.shape)
# full_clusters[:10]

### 2) Nearest-Neighbor Index

Row index into `labeled_rows` for vectorized similarity lookup. Each labeled image is matched
against the full corpus embedding matrix by cosine similarity.


In [ ]:
row_idx = np.arange(len(labeled_rows))

# row_idx[:10]
# len(row_idx)

### 3) Nearest Non-Self CLIP Match

Both labeled and full-corpus embeddings are L2-normalized before the dot product. `argmax`
gives the single nearest neighbor per labeled row. Similarity scores below `SIM_MIN` (0.98) are
discarded--only near-duplicate matches are used as votes, not approximate neighbors.


In [ ]:
labeled_norm = np.linalg.norm(labeled, axis=1, keepdims=True)
full_norm    = np.linalg.norm(full, axis=1, keepdims=True)
labeled_unit = labeled / np.clip(labeled_norm, 1e-12, None)
full_unit    = full / np.clip(full_norm, 1e-12, None)
sim          = labeled_unit @ full_unit.T

row_idx    = np.arange(len(labeled_rows))
nn_idx     = sim.argmax(axis=1)
nn_sim     = sim[row_idx, nn_idx]
mapped_cid = full_clusters[nn_idx]

# (sim.shape, row_idx.shape, nn_idx.shape, nn_sim.shape, mapped_cid.shape)
# nn_sim[:10]

### 4) Vote Aggregation

Valid matches are grouped by cluster id. For each cluster, label counts across matched labeled
rows are normalized by cluster size to produce vote share. The result is `clip_vote`: a
cluster × label matrix where each cell is the fraction of matched rows in that cluster carrying
a given label.


In [ ]:
mapped = [
    (i, int(cid))
    for i, cid in enumerate(mapped_cid)
    if int(cid) != -1 and float(nn_sim[i]) >= SIM_MIN
]

cluster_sizes = Counter(cid for _, cid in mapped)
label_counts  = {}
for row_idx, cid in mapped:
    label_counts.setdefault(cid, Counter()).update(labeled_rows[row_idx]["categories"])

labels    = sorted({label for counts in label_counts.values() for label in counts})
clip_vote = pd.DataFrame(0.0, index=sorted(label_counts), columns=labels)
for cid, counts in label_counts.items():
    denom = cluster_sizes[cid]
    for label, count in counts.items():
        clip_vote.loc[cid, label] = count / denom

# len(mapped)
# clip_vote.shape
# pd.Series(cluster_sizes).sort_values(ascending=False).head(10)

### 5) Filter Matrix

Row and column order inherited from the OCR-Jaccard section (`ROW_ORDER`, `COL_ORDER`) so both
heatmaps are directly comparable. Dominant label per row is the highest vote-share label.


In [ ]:
C        = clip_vote.reindex(index=ROW_ORDER, columns=COL_ORDER, fill_value=0).copy()
dominant = C.idxmax(axis=1).where(C.max(axis=1) > 0,
                                  "none") if COL_ORDER else pd.Series("none", index=C.index)
C.index  = [f"c{cid:>3} n={cluster_sizes[cid]:>2} [{dominant.loc[cid]}]" for cid in C.index]

# (len(COL_ORDER), len(ROW_ORDER), C.shape)
# C.iloc[:5, :5]

### 6) Plot Focused CLIP Vote Clustermap

In [ ]:
vmax  = np.percentile(C.values, VMAX_PCTILE)
fig_w = max(14, len(COL_ORDER) * 0.45)
fig_h = max(12, len(ROW_ORDER) * 0.22)
cg    = sns.clustermap(
    C,
    method="ward",
    metric="euclidean",
    cmap="YlOrRd",
    vmin=0,
    vmax=vmax,
    figsize=(fig_w, fig_h),
    linewidths=0.2,
    linecolor="#e0e0e0",
    xticklabels=True,
    yticklabels=True,
    row_cluster=False,
    col_cluster=False,
    dendrogram_ratio=(0.10, 0.06),
    cbar_pos=(0.01, 0.82, 0.02, 0.14),
    cbar_kws={"label": f"CLIP vote share (capped p{VMAX_PCTILE})"},
)
# check: (C.shape, vmax, fig_w, fig_h)
plt.setp(cg.ax_heatmap.get_xticklabels(), rotation=45, ha="right", fontsize=8)
plt.setp(cg.ax_heatmap.get_yticklabels(), fontsize=7.5)
cg.ax_heatmap.set_xlabel("Label", fontsize=10)
cg.ax_heatmap.set_ylabel("")
cg.ax_col_dendrogram.set_title(
    "CLIP Cluster x Label Vote (focused) - Ward/Euclidean",
    fontsize=11,
)
plt.show()

## CLIP Vote vs OCR-Jaccard

Both methods are evaluated on the same 171 mapped rows using a leave-one-out protocol. Neither
method sees the row being predicted--peers or leave-out exclusion enforced in both cases.

`clip_preds`: collects labels from peer rows in the same cluster, keeps labels exceeding
`min_share` of peers. `ocr_preds`: builds per-label OCR token vocabularies from other rows,
scores each label against the cluster's OCR vocab by Jaccard, keeps top-k above `min_score`.
`score` converts prediction sets to binary matrices and reports coverage, hit-rate, micro/macro
F1, and per-label F1.


### 1) Prepare Shared Evaluation Rows and Grouped OCR Tokens

In [ ]:
eval_rows = [(row_idx, cid, set(labeled_rows[row_idx]["categories"])) for row_idx, cid in mapped]
tokens    = [tokenize(labeled_text_ocr[row_idx]) for row_idx, _, _ in eval_rows]

by_cluster = {}
for pos, (_, cid, _) in enumerate(eval_rows):
    by_cluster.setdefault(cid, []).append(pos)

mapped_cluster_vocab = {}
for pos, (_, cid, _) in enumerate(eval_rows):
    mapped_cluster_vocab.setdefault(cid, set()).update(tokens[pos])

# len(eval_rows), len(tokens), len(by_cluster), len(mapped_cluster_vocab)
# pd.Series([len(v) for v in by_cluster.values()]).describe()

### 2) Define CLIP Vote Baseline
`clip_preds` baseline: for each row, collect labels from peer rows in the same cluster and keep labels that exceed `min_share`.

In [ ]:
def clip_preds(min_peers=2, top_k=3, min_share=0.40):
    preds = []
    for pos, (_, cid, _) in enumerate(eval_rows):
        peers = [j for j in by_cluster[cid] if j != pos]
        if len(peers) < min_peers:
            preds.append(set())
            continue
        counts = Counter(label for j in peers for label in eval_rows[j][2])
        ranked = sorted(counts.items(), key=lambda item: item[1], reverse=True)[:top_k]
        preds.append({label for label, count in ranked if (count / len(peers)) >= min_share})
    return preds


# preview_clip = clip_preds()
# [sorted(list(p)) for p in preview_clip[:5]]

### 3) Define OCR-Jaccard Baseline

`ocr_preds` baseline: build label-specific token vocab from other rows, compare each label vocab with mapped-cluster OCR vocab (same eval rows) using Jaccard similarity.

In [ ]:
def ocr_preds(top_k=3, min_score=0.10):
    preds = []
    for pos, (_, cid, _) in enumerate(eval_rows):
        label_vocab = {}
        for j, (_, _, true_labels) in enumerate(eval_rows):
            if j == pos:
                continue
            for label in true_labels:
                label_vocab.setdefault(label, set()).update(tokens[j])
        c_tokens = mapped_cluster_vocab.get(cid, set())
        ranked = sorted(
            ((label, jaccard(c_tokens, tks)) for label, tks in label_vocab.items()),
            key=lambda item: item[1],
            reverse=True,
        )[:top_k]
        preds.append({label for label, score in ranked if score >= min_score})
    return preds


# preview_ocr = ocr_preds()
# [sorted(list(p)) for p in preview_ocr[:5]]

### 4) Define Scoring Function

`score` converts label sets to binary matrices and reports coverage, hit-rate, micro/macro F1, and per-label F1 deltas.

In [ ]:
def score(method, preds):
    y_true_list = [sorted(labels) for _, _, labels in eval_rows]
    y_pred_list = [sorted(p) for p in preds]
    mlb = MultiLabelBinarizer()
    y_true = mlb.fit_transform(y_true_list)
    y_pred = mlb.transform(y_pred_list)
    metrics = {
        "method": method,
        "n": len(eval_rows),
        "coverage": sum(len(p) > 0 for p in preds) / len(eval_rows),
        "hit_rate": sum(len(p & t) > 0 for p, t in zip(preds, [r[2] for r in eval_rows]))
        / len(eval_rows),
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "mean_pred_labels": float(np.mean([len(p) for p in preds])),
    }
    per_label = pd.Series(
        f1_score(y_true, y_pred, average=None, zero_division=0),
        index=mlb.classes_,
        dtype=float,
    )
    support = pd.Series(y_true.sum(axis=0), index=mlb.classes_, dtype=float)
    return metrics, per_label, support


# debug_metrics, debug_f1, debug_support = score("debug", clip_preds())
# debug_metrics
# debug_support.sort_values(ascending=False).head(10)

### 5) Run Both Predictors on the Same Rows

In [ ]:
pred_clip = clip_preds()
pred_ocr  = ocr_preds()

# (len(pred_clip), len(pred_ocr))
# (pred_clip[:3], pred_ocr[:3])

### Results

```
                n  coverage  hit_rate  micro_f1  macro_f1  mean_pred_labels
CLIP vote     171     0.608     0.415     0.343     0.116             1.310
OCR-Jaccard   171     0.889     0.526     0.324     0.164             2.398
```

Neither method is dominant. CLIP vote is more precise--higher micro F1, lower mean predicted labels per row (1.3 vs 2.4), fewer false positives. OCR-Jaccard covers more rows (89% vs 61%) and achieves higher hit-rate and macro F1, meaning it reaches more labels including low-support ones.

The per-label breakdown clarifies where each method works:

CLIP vote wins on visually distinctive, text-light labels - `browser` (+0.51), `screenshare` (+0.35), `webdev` (+0.26), `receipt` (+0.15), `hockey` (+0.04). These are categories where visual structure is the primary discriminator and OCR is sparse or noisy.

OCR-Jaccard wins on text-dense, lexically specific labels - `plex` (+1.00), `hash` (+1.00),
`dev tools` (+0.80), `map` (+0.57), `graphs` (+0.57). These are categories where a specific vocabulary is strongly predictive and visual appearance is either generic or variable.

The synthesis F1 comparison (cell 45) confirms this split at the classifier level: image micro 0.376 vs text 0.304, image macro 0.224 vs text 0.080. CLIP is the stronger default representation, but OCR carries signal that CLIP misses entirely for text-anchored categories. The two are complementary, not redundant.


In [ ]:
m_clip, f1_clip, support = score("CLIP vote", pred_clip)
m_ocr, f1_ocr, _       = score("OCR-Jaccard", pred_ocr)

cmp = pd.DataFrame([m_clip, m_ocr]).set_index("method")
print(
    cmp[["n", "coverage", "hit_rate", "micro_f1", "macro_f1", "mean_pred_labels"]].to_string(
        float_format="{:.3f}".format
    )
)

delta     = (f1_clip - f1_ocr).rename("clip_minus_ocr")
label_cmp = pd.concat(
    [f1_clip.rename("clip_f1"), f1_ocr.rename("ocr_f1"), delta, support.rename("support")],
    axis=1,
).sort_values("clip_minus_ocr", ascending=False)

# cmp
# label_cmp.head(10)

#### Top Labels Where CLIP > OCR-Jaccard:

In [ ]:
print(label_cmp.head(10).to_string(float_format="{:.3f}".format))

#### Top Labels Where OCR-Jaccard >= CLIP:

In [ ]:
print(label_cmp.tail(10).to_string(float_format="{:.3f}".format))

In [ ]:
stopwatch(0)
# 129.23865489498712 # (seconds)